# Hiltunen Long-Format Preprocessing

## 1. Objective
Convert the experimental Hiltunen daily cell-density and pH measurements from wide to long format. Keep each experiment/funnel identity and all day rows, including missing measurements. This phase does not create prediction features, train models, or claim that the source directly represents the Liquid Tree prototype.

In [ ]:
## 2. Load raw dataset
## 3. Identify metadata and daily time-series columns
## 4. Inspect source dimensions
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

WORKBOOK_RELATIVE_PATH = Path("data/raw/hiltunen/hiltunen_etal_chlorella_growth_biochem_data.xlsx")
search_locations = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next(
    (location for location in search_locations if (location / WORKBOOK_RELATIVE_PATH).is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"Could not find {WORKBOOK_RELATIVE_PATH} from this directory or its parents.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.preprocessing.hiltunen_long_format as hiltunen_preprocessing

hiltunen_preprocessing = importlib.reload(hiltunen_preprocessing)
SOURCE_SHEET = hiltunen_preprocessing.SOURCE_SHEET
build_metadata = hiltunen_preprocessing.build_metadata
identify_source_columns = hiltunen_preprocessing.identify_source_columns
missingness_report = hiltunen_preprocessing.missingness_report
reshape_to_long = hiltunen_preprocessing.reshape_to_long
validate_transformation = hiltunen_preprocessing.validate_transformation

source_path = PROJECT_ROOT / WORKBOOK_RELATIVE_PATH
source = pd.read_excel(source_path, sheet_name=SOURCE_SHEET)
metadata_columns, cell_columns, ph_columns = identify_source_columns(source)
print(f"Source sheet: {SOURCE_SHEET!r}")
print(f"Source dimensions: {source.shape[0]} rows x {source.shape[1]} columns")
print(f"Metadata columns: {metadata_columns}")
print(f"Detected cell-density columns ({len(cell_columns)}): {list(cell_columns.values())}")
print(f"Detected pH columns ({len(ph_columns)}): {list(ph_columns.values())}")
display(source.head())

## 5. Wide-to-long transformation
Each source row contributes one row per detected day. `CO2` and other metadata labels are retained exactly as stored; no numeric CO₂ conversion is attempted.

In [ ]:
long_data = reshape_to_long(source, metadata_columns, cell_columns, ph_columns)
validate_transformation(source, long_data, metadata_columns, cell_columns, ph_columns)
print(f"Long-format dimensions: {long_data.shape[0]} rows x {long_data.shape[1]} columns")
print("Output columns:", long_data.columns.tolist())
display(long_data.head(10))

## 6. Missingness analysis
No values are filled, interpolated, replaced with zero, or removed. The paired-complete count is the number of rows per day with both cell density and pH observed.

In [ ]:
day_report = missingness_report(long_data)
display(day_report)

sns.set_theme(style="whitegrid", context="notebook")
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
sns.barplot(data=day_report, x="Day", y="Missing_Cell_Density", color="#b45f45", ax=axes[0])
axes[0].set(title="Missing cell-density measurements by day", xlabel="Day", ylabel="Missing observations (count)")
sns.histplot(data=long_data, x="Cell_Density", bins=25, kde=False, color="#287a78", ax=axes[1])
axes[1].set(title="Observed cell-density distribution", xlabel="Cell density (10^6 cells/mL)", ylabel="Observations (count)")
plt.show()

print("A pooled mean/median trajectory is not plotted here: later days have substantially fewer observed cultures,")
print("so the set of cultures contributing to a day-level summary changes over time.")

## 7. Transformation validation
The reusable validator checks every source density and pH value, missingness, identity counts, day coverage, expected row count, and duplicate experiment/funnel/day keys. The next cell also compares a reproducibly selected subset for easy inspection.

In [ ]:
validate_transformation(source, long_data, metadata_columns, cell_columns, ph_columns)
print(f"Unique experiments: {source['Exp'].nunique()} source; {long_data['Exp'].nunique()} long")
print(f"Unique funnels: {source['Funnel#'].nunique()} source; {long_data['Funnel#'].nunique()} long")
print(f"Day range: {long_data['Day'].min()} to {long_data['Day'].max()}")
print("Duplicate experiment/funnel/day records:", int(long_data.duplicated(['Exp', 'Funnel#', 'Day']).sum()))

sampled_sources = source.sample(n=min(3, len(source)), random_state=17)
spot_checks = []
for _, source_row in sampled_sources.iterrows():
    for day in [0, 8, 16]:
        long_row = long_data.loc[
            (long_data['Exp'] == source_row['Exp'])
            & (long_data['Funnel#'] == source_row['Funnel#'])
            & (long_data['Day'] == day)
        ].iloc[0]
        assert (pd.isna(long_row['Cell_Density']) and pd.isna(source_row[cell_columns[day]])) or long_row['Cell_Density'] == source_row[cell_columns[day]]
        assert (pd.isna(long_row['pH']) and pd.isna(source_row[ph_columns[day]])) or long_row['pH'] == source_row[ph_columns[day]]
        spot_checks.append({
            'Exp': source_row['Exp'],
            'Funnel#': source_row['Funnel#'],
            'Day': day,
            'Cell_Density': long_row['Cell_Density'],
            'pH': long_row['pH'],
        })
display(pd.DataFrame(spot_checks))
print("Full round-trip validation and selected source-to-long checks passed.")

## 8. Basic descriptive inspection
These summaries describe observed measurements only. The observation count is shown because culture availability differs across days; these summaries do not imply that each day represents the same set of cultures.

In [ ]:
display(
    long_data.groupby("Day", sort=True)["Cell_Density"]
    .agg(observed_count="count", mean="mean", median="median", minimum="min", maximum="max")
    .reset_index()
)
print("Cell density is documented in the source as 10^6 cells/mL. No trajectory aggregation is interpreted as a model result.")

## 9. Export processed dataset
Write the long-format CSV and machine-readable processing metadata. The raw workbook is only read.

In [ ]:
import json

from src.preprocessing.hiltunen_long_format import (
    METADATA_RELATIVE_PATH,
    OUTPUT_RELATIVE_PATH,
)

csv_path = PROJECT_ROOT / OUTPUT_RELATIVE_PATH
metadata_path = PROJECT_ROOT / METADATA_RELATIVE_PATH
csv_path.parent.mkdir(parents=True, exist_ok=True)
long_data.to_csv(csv_path, index=False, na_rep="")
metadata = build_metadata(
    source_path, source.shape, long_data, metadata_columns, cell_columns, ph_columns
)
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
reloaded = pd.read_csv(csv_path)
pd.testing.assert_frame_equal(long_data, reloaded, check_dtype=False, check_names=True)
assert reloaded["Day"].min() == 0 and reloaded["Day"].max() == 16
assert not reloaded.duplicated(["Exp", "Funnel#", "Day"]).any()
print(f"Saved and reloaded {reloaded.shape[0]} rows x {reloaded.shape[1]} columns: {csv_path.relative_to(PROJECT_ROOT)}")
print(f"Metadata: {metadata_path.relative_to(PROJECT_ROOT)}")
print("CSV round-trip, day coverage, and duplicate-key checks passed.")